# Credit Risk Scoring Engine — Data Preprocessing & Leakage Audit

## Overview & Objectives
This notebook establishes a production-grade data preprocessing pipeline for the German Credit Risk dataset (`data/raw/german_credit.csv`).

Key tasks accomplished in this stage:
1. **Target Identification & Mapping**: Binary encoding of `credit_risk` (`1` -> `0` Good Credit, `2` -> `1` Bad Credit / Default).
2. **Data Integrity Verification**: Verification of missing values, duplicates, and feature data types.
3. **Data Leakage Audit**: Systematic audit of all 20 predictive attributes to ensure no post-outcome or target-derived information leaks into training.
4. **Stratified Train/Test Split**: 80/20 stratified partitioning performed **BEFORE** fitting transformations.
5. **Scikit-Learn Pipeline & ColumnTransformer**: Encoding categorical features via `OneHotEncoder(handle_unknown='ignore')` and scaling numerical features via `StandardScaler()`.
6. **Artifact Export**: Reproducible serialization of preprocessor to `models/preprocessing_pipeline.joblib` and datasets to `data/processed/`.

In [2]:
import os
import json
import joblib
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

# Load raw dataset
raw_data_path = '../data/raw/german_credit.csv'
df = pd.read_csv(raw_data_path)
print(f"Raw Dataset Loaded: {df.shape[0]} rows, {df.shape[1]} columns")
print("Columns:", list(df.columns))


In [3]:
# Raw credit_risk: 1 = Good, 2 = Bad
# Map to standard binary target: 0 = Good Credit (Non-Default), 1 = Bad Credit (Default)
target_map = {1: 0, 2: 1}
df['target'] = df['credit_risk'].map(target_map)

print("=== Target Distribution Verification ===")
counts = df['target'].value_counts()
rates = df['target'].value_counts(normalize=True) * 100
for val in [0, 1]:
    label = "Good Credit (0)" if val == 0 else "Bad Credit / Default (1)"
    print(f"  {label}: {counts[val]} applicants ({rates[val]:.2f}%)")

print(f"Total Observations: {len(df)} | Total Defaults: {df['target'].sum()} | Default Rate: {df['target'].mean()*100:.2f}%")


In [4]:
print("=== Data Integrity Checks ===")
print(f"Duplicate Rows: {df.duplicated().sum()}")
missing = df.isnull().sum()
print(f"Missing Values: {missing.sum()}")

numerical_cols = ['duration_in_months', 'credit_amount', 'installment_rate', 
                  'present_residence_since', 'age_in_years', 'existing_credits', 'num_people_liable']
categorical_cols = ['status_checking_account', 'credit_history', 'purpose', 'savings_account',
                    'present_employment_since', 'personal_status_sex', 'other_debtors_guarantors',
                    'property', 'other_installment_plans', 'housing', 'job', 'telephone', 'foreign_worker']

print(f"Numerical Features Count:   {len(numerical_cols)}")
print(f"Categorical Features Count: {len(categorical_cols)}")
print(f"Total Predictors:          {len(numerical_cols) + len(categorical_cols)}")


## Data Leakage Audit

Data leakage occurs when information from outside the training dataset or post-outcome information is used to fit the model. In credit risk modeling, common leakage vectors include:
1. **Post-Outcome / Collection Features**: Variables captured after loan issuance (e.g., late payment counts during the loan term, charge-off indicators).
2. **Target-Derived Variables**: Features engineered using target statistics across the full dataset prior to train/test splitting.
3. **Split Contamination**: Fitting scalers, encoders, or imputers on the full dataset before splitting.
4. **Duplicate Record Leakage**: Identical applicant records appearing in both training and test sets.

### Systematic Feature Evaluation

| Feature | Timing / Source | Decision | Justification |
| :--- | :--- | :---: | :--- |
| `status_checking_account` | Application Time | **KEEP** | Account liquidity status at application time. |
| `duration_in_months` | Loan Application | **KEEP** | Requested loan term. Captured prior to origination. |
| `credit_history` | Bureau History | **KEEP** | Historical repayment track record prior to application. |
| `purpose` | Loan Application | **KEEP** | Intended loan purpose stated at application. |
| `credit_amount` | Loan Application | **KEEP** | Requested principal amount. |
| `savings_account` | Application Time | **KEEP** | Liquid savings balance at application time. |
| `present_employment_since` | Applicant Demographics | **KEEP** | Employment tenure at application date. |
| `installment_rate` | Application Time | **KEEP** | Debt-service burden (% of disposable income). |
| `personal_status_sex` | Applicant Demographics | **KEEP** | Demographic information captured at origination. |
| `other_debtors_guarantors` | Credit Application | **KEEP** | Co-signers/guarantors present at origination. |
| `present_residence_since` | Applicant Demographics | **KEEP** | Residential stability at application date. |
| `property` | Financial Disclosure | **KEEP** | Asset collateral disclosed at application. |
| `age_in_years` | Applicant Demographics | **KEEP** | Applicant age at application date. |
| `other_installment_plans` | Financial Disclosure | **KEEP** | Existing external debt obligations. |
| `housing` | Applicant Demographics | **KEEP** | Housing tenure status (own/rent/free). |
| `existing_credits` | Credit Bureau | **KEEP** | Number of active credits at this bank. |
| `job` | Employment Disclosure | **KEEP** | Occupational qualification tier. |
| `num_people_liable` | Family Disclosure | **KEEP** | Dependents count disclosed at application. |
| `telephone` | Contact Info | **KEEP** | Phone registration status. |
| `foreign_worker` | Regulatory Info | **KEEP** | Residency/worker status. |

> [!IMPORTANT]
> **Audit Conclusion**: All 20 predictor attributes represent historical information available at loan application origination. No post-outcome debt collection features or target-derived leakage variables exist in the raw dataset.

In [6]:
X = df[numerical_cols + categorical_cols].copy()
y = df['target'].copy()

# Perform Stratified Train/Test Split BEFORE fitting any transformation
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    stratify=y,
    random_state=42
)

print("=== Train / Test Split Summary ===")
print(f"Training Set: {X_train.shape[0]} rows (Good: {(y_train==0).sum()}, Bad: {(y_train==1).sum()}, Default Rate: {y_train.mean()*100:.2f}%)")
print(f"Test Set:     {X_test.shape[0]} rows (Good: {(y_test==0).sum()}, Bad: {(y_test==1).sum()}, Default Rate: {y_test.mean()*100:.2f}%)")


In [7]:
# Numeric transformer: StandardScaler
numeric_transformer = Pipeline(steps=[
    ('scaler', StandardScaler())
])

# Categorical transformer: OneHotEncoder
categorical_transformer = Pipeline(steps=[
    ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

# Combine into ColumnTransformer
preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numerical_cols),
        ('cat', categorical_transformer, categorical_cols)
    ],
    remainder='drop'
)

# Fit preprocessor STRICTLY on X_train
preprocessor.fit(X_train)

# Get feature names out from ColumnTransformer
feature_names_out = list(preprocessor.get_feature_names_out())

print(f"Preprocessing Pipeline Fitted Successfully!")
print(f"Total Transformed Feature Count: {len(feature_names_out)}")
print("Sample Transformed Features:", feature_names_out[:10])


In [8]:
# Transform train and test sets
X_train_prep = preprocessor.transform(X_train)
X_test_prep = preprocessor.transform(X_test)

X_train_df = pd.DataFrame(X_train_prep, columns=feature_names_out)
X_test_df = pd.DataFrame(X_test_prep, columns=feature_names_out)

print(f"X_train_prep shape: {X_train_prep.shape}")
print(f"X_test_prep shape:  {X_test_prep.shape}")

# Check for NaN or Inf
assert not np.isnan(X_train_prep).any(), "NaN found in X_train_prep!"
assert not np.isnan(X_test_prep).any(), "NaN found in X_test_prep!"
print("Data integrity assertion passed: No NaNs or Infs in transformed features.")


In [9]:
# Ensure output directories exist
os.makedirs('../models', exist_ok=True)
os.makedirs('../data/processed', exist_ok=True)

# Save preprocessor pipeline
joblib.dump(preprocessor, '../models/preprocessing_pipeline.joblib')
print("Saved preprocessor to ../models/preprocessing_pipeline.joblib")

# Save feature names metadata
metadata = {
    'numerical_cols': numerical_cols,
    'categorical_cols': categorical_cols,
    'transformed_feature_names': feature_names_out,
    'num_features_raw': len(numerical_cols) + len(categorical_cols),
    'num_features_transformed': len(feature_names_out),
    'train_samples': len(X_train),
    'test_samples': len(X_test)
}
with open('../models/feature_metadata.json', 'w') as f:
    json.dump(metadata, f, indent=2)
print("Saved metadata to ../models/feature_metadata.json")

# Save processed DataFrames
X_train_df.to_csv('../data/processed/X_train.csv', index=False)
X_test_df.to_csv('../data/processed/X_test.csv', index=False)
y_train.to_csv('../data/processed/y_train.csv', index=False)
y_test.to_csv('../data/processed/y_test.csv', index=False)

# Save combined processed datasets for exploration
train_proc = X_train_df.copy()
train_proc['target'] = y_train.values
train_proc.to_csv('../data/processed/train_processed.csv', index=False)

test_proc = X_test_df.copy()
test_proc['target'] = y_test.values
test_proc.to_csv('../data/processed/test_processed.csv', index=False)

print("Saved all processed datasets to ../data/processed/")
